In [ ]:
#short-term memory for the agent’s internal trajectory

class Memory:
    def __init__(self):
        self.records = []

    def add_record(self,record_type,content):
        record = {
            "type":record_type,
            "content": content
        }
        self.records.append(record)

    def get_record(self):
        return self.records

    def get_record_by_type(self,record_type):
        result = []
        for record in self.records:
            if record["type"] == record_type:
                result.append(record)
        return result

    def get_trajectory(self):
        trajectory = []
        for record in self.records:
            if record["type"] == "execution":
                trajectory.append(
                    f"Execution:\n{record['content']}"
                )
            elif record["type"] == "reflection":
                trajectory.append(
                    f"Reflection:\n{record['content']}"
                )

        return "\n\n".join(trajectory)

    def get_last_execution(self):
        #`reversed()` iterates through a list in reverse order.
        for record in reversed(self.records):
            if record["type"]=="execution":
                return record["content"]
        return None
        

In [ ]:
INITIAL_PROMPT_TEMPLATE = """
You are a senior Python programmer. Please write a Python function according to the following requirements.
Your code must include a complete function signature, docstring, and follow PEP 8 coding standards.

Requirement: {task}

Please output the code directly without any additional explanations.
"""


In [ ]:
REFLECT_PROMPT_TEMPLATE = """
You are an extremely strict code review expert and senior algorithm engineer with ultimate requirements for code performance.
Your task is to review the following Python code and focus on finding its main bottlenecks in <strong>algorithm efficiency</strong>.

# Original Task:
{task}

# Code to Review:
```python
{code}
```

Please analyze the time complexity of this code and consider whether there is an <strong>algorithmically superior</strong> solution to significantly improve performance.
If one exists, please clearly point out the deficiencies of the current algorithm and propose specific, feasible algorithm improvement suggestions (e.g., using sieve method instead of trial division).
Only if the code has reached optimality at the algorithm level can you answer "no improvement needed."

Please output your feedback directly without any additional explanations.
"""


In [ ]:

REFINE_PROMPT_TEMPLATE = """
You are a senior Python programmer. You are optimizing your code based on feedback from a code review expert.

# Original Task:
{task}

# Your Previous Code Attempt:
{last_code_attempt}
Reviewer's Feedback:
{feedback}

Please generate an optimized new version of the code based on the reviewer's feedback.
Your code must include a complete function signature, docstring, and follow PEP 8 coding standards.
Please output the optimized code directly without any additional explanations.
"""


In [ ]:
from llm_client import HelloAgentsLLM

class ReflectionAgent:
    def __init__(self,llm_client,max_iterations=3):
        self.llm = llm_client
        self.memory = Memory()
        self.iterations = max_iterations

    def run(self,task):
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(
            task=task
            )

        first_response = self._get_llm_response(
            initial_prompt
        )

        self.memory.add_record(
            "execution",
            first_response
        )
        for i in range(self.iterations):
            last_code = self.memory.get_last_execution()

            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(
                task = task,
                code = last_code
            )

            feedback = self._get_llm_response(
                reflect_prompt
            )

            self.memory.add_record(
                        "reflection",
                        feedback
                    )
            if "no improvement needed" in feedback.lower():
                print("\n✅ Reflection considers code needs no improvement, task completed.")
                break

            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task = task,
                last_code_attempt = last_code,
                feedback = feedback
            )
            result = self._get_llm_response(
                refine_prompt
            )
            self.memory.add_record(
                        "execution",
                        result
                    )

        return self.memory.get_last_execution()

    def _get_llm_response(self, prompt):
        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        response = self.llm.think(
            messages=messages
        )
        
        print(f"\n--- Task Completed ---\nFinal Generated Code:\n```python\n{final_code}\n```")

        return response

